<a href="https://colab.research.google.com/github/Joyash22/Abhilasha---Multi-Agent-Ai-medical-assistant/blob/main/Abhilasha_Medical_QLoRA_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Abhilasha — Medical QA Fine-Tune with QLoRA
### Fine-tuning Mistral-7B-Instruct on Medical QA | by Joyash Atri

**What this notebook does:**
- Loads `Mistral-7B-Instruct-v0.2` in **4-bit NF4 quantisation** (fits on a free T4 GPU)
- Applies **QLoRA** (LoRA adapters on attention layers) — trains only ~8M of 7B parameters
- Fine-tunes on **MedAlpaca medical QA** dataset
- Evaluates before/after perplexity on a held-out test split
- Pushes the adapter weights to **Hugging Face Hub**

**Runtime:** ~90 min total on Colab T4  
**Cost:** ₹0



## 0 · Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")


Sat Jun 27 08:07:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1 · Install Dependencies




In [ ]:
%%capture
!pip install -U bitsandbytes==0.43.3
!pip install -U transformers==4.42.4
!pip install -U peft==0.11.1
!pip install -U accelerate==0.31.0
!pip install -U trl==0.9.4
!pip install -U datasets==2.20.0
!pip install -U huggingface_hub
!pip install -U sentencepiece
!pip install -U evaluate

print("✅ All packages installed")


## 2 · Imports & Config

In [ ]:
!pip install -q numpy==1.26.4 --force-reinstall
!pip install -q transformers==4.41.0 datasets==2.19.0 peft==0.11.1 trl==0.9.4 accelerate==0.30.0 bitsandbytes==0.43.1 huggingface_hub sentencepiece evaluate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.42.4 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.21.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask

In [ ]:
import os
import torch
import json
import math
from datetime import datetime

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging as hf_logging,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer
from huggingface_hub import login

hf_logging.set_verbosity_error()

# Configuration
BASE_MODEL    = "mistralai/Mistral-7B-Instruct-v0.2"
NEW_MODEL     = "abhilasha-medical-mistral-7b-qlora"
HF_USERNAME   = "joyash"
DATASET_NAME  = "medalpaca/medical_meadow_medqa"
MAX_SEQ_LEN   = 512
TRAIN_SAMPLES = 3000
TEST_SAMPLES  = 300

print(f"Base model  : {BASE_MODEL}")
print(f"New model   : {HF_USERNAME}/{NEW_MODEL}")
print(f"Dataset     : {DATASET_NAME}")
print(f"Train size  : {TRAIN_SAMPLES} samples")
print(f"Device      : {torch.cuda.get_device_name(0)}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 3 · Hugging Face Login



In [ ]:
# Option A: Colab Secrets (recommended — doesn't expose token in notebook)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ Loaded token from Colab Secrets")
except Exception:
    # Option B: Paste directly (remove before sharing notebook)
    HF_TOKEN = "hf_YOUR_TOKEN_HERE"
    print("⚠️  Using hardcoded token — remove before sharing")

login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged in to Hugging Face Hub")


## 4 · Load & Format Dataset

**MedAlpaca Medical Meadow MedQA** — US Medical Licensing Exam (USMLE) style questions.

Each sample has:
- `input`: a clinical question
- `output`: the correct answer with explanation

We format into Mistral's instruction template: `<s>[INST] ... [/INST] ... </s>`


In [ ]:
# Load dataset
raw = load_dataset(DATASET_NAME, trust_remote_code=True)
print(f"Raw dataset: {raw}")
print(f"\nSample row:\n{raw['train'][0]}")


In [ ]:
SYSTEM_PROMPT = """You are Abhilasha, a precise and compassionate AI medical assistant. \
Answer clinical questions accurately, cite relevant medical reasoning, and always recommend \
professional consultation for personal medical decisions.""".strip()

def format_prompt(row):
    """Format a dataset row into Mistral instruction template."""
    instruction = row.get("instruction", "").strip()
    inp         = row.get("input", "").strip()
    output      = row.get("output", "").strip()

    # Combine instruction + input if both present
    question = f"{instruction}\n\n{inp}" if instruction and inp else (instruction or inp)

    return {
        "text": (
            f"<s>[INST] {SYSTEM_PROMPT}\n\n"
            f"Question: {question} [/INST]\n"
            f"{output}</s>"
        )
    }

# Apply formatting
formatted = raw["train"].map(format_prompt, remove_columns=raw["train"].column_names)

# Filter out very short samples (likely malformed)
formatted = formatted.filter(lambda x: len(x["text"]) > 100)

# Split into train / test
total = len(formatted)
n_train = min(TRAIN_SAMPLES, int(total * 0.9))
n_test  = min(TEST_SAMPLES,  total - n_train)

train_ds = formatted.select(range(n_train))
test_ds  = formatted.select(range(n_train, n_train + n_test))

print(f"Train samples : {len(train_ds)}")
print(f"Test samples  : {len(test_ds)}")
print(f"\nFormatted example (first 400 chars):")
print(train_ds[0]["text"][:400])


## 5 · Load Base Model & Measure Baseline Perplexity

We measure perplexity **before** fine-tuning so we have a real before/after number.

**Perplexity** = how "surprised" the model is by the test text.  
Lower = better. A well-tuned domain model should score lower than the base model.


In [ ]:
# 4-bit quantisation config — this is the core of QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — best quality at 4-bit
    bnb_4bit_use_double_quant=True,      # quantise the quantisation constants too
    bnb_4bit_compute_dtype=torch.float16 # compute in fp16 for speed
)

print("Loading base model (this takes ~3-4 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False           # required for gradient checkpointing
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"         # Mistral needs right-padding

print(f"✅ Model loaded")
print(f"   Parameters : {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"   VRAM used  : {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
def compute_perplexity(mdl, tokenizer, dataset, n_samples=50, max_len=256):
    """Compute mean perplexity over n_samples from dataset."""
    mdl.eval()
    losses = []
    samples = dataset.select(range(min(n_samples, len(dataset))))

    with torch.no_grad():
        for item in samples:
            inputs = tokenizer(
                item["text"],
                return_tensors="pt",
                truncation=True,
                max_length=max_len,
            ).to("cuda")

            # labels = input_ids so loss is next-token prediction
            outputs = mdl(**inputs, labels=inputs["input_ids"])
            losses.append(outputs.loss.item())

    mean_loss = sum(losses) / len(losses)
    return math.exp(mean_loss), mean_loss

print("Computing baseline perplexity on 50 test samples...")
print("(~2 minutes)")
baseline_ppl, baseline_loss = compute_perplexity(model, tokenizer, test_ds, n_samples=50)
print(f"\n📊 Baseline perplexity : {baseline_ppl:.2f}")
print(f"   Baseline NLL loss   : {baseline_loss:.4f}")
print("\nSave these numbers — you'll compare after fine-tuning.")


## 6 · Test Base Model Response (Before Fine-Tune)

In [ ]:
TEST_QUESTION = (
    "A 45-year-old male presents with chest pain radiating to the left arm, "
    "diaphoresis, and shortness of breath for the past 2 hours. "
    "His ECG shows ST-segment elevation in leads II, III, and aVF. "
    "What is the most likely diagnosis and immediate management?"
)

def ask_model(mdl, tok, question, max_new_tokens=300):
    prompt = (
        f"<s>[INST] {SYSTEM_PROMPT}\n\n"
        f"Question: {question} [/INST]\n"
    )
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

print("=" * 70)
print("QUESTION:", TEST_QUESTION)
print("=" * 70)
print("\nBASE MODEL RESPONSE:")
base_response = ask_model(model, tokenizer, TEST_QUESTION)
print(base_response)
print("=" * 70)
print("\n⬆️  Save this output — you'll compare it to the fine-tuned response.")


## 7 · Configure LoRA Adapters

**What LoRA does:** Instead of updating all 7B weights, we inject small trainable  
rank-decomposition matrices (A and B) alongside the frozen attention layers.  
Only ~0.1% of parameters are trained — but the model learns your domain.

Key hyperparameters:
- `r=16` — LoRA rank. Higher = more capacity but more memory. 16 is the sweet spot for T4.
- `lora_alpha=32` — scaling factor (keep it 2× rank)
- `target_modules` — which layers to inject adapters into. q/v projections = most impactful.


In [ ]:
# Prepare model for k-bit training (adds gradient checkpointing etc.)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",    # query projection
        "k_proj",    # key projection
        "v_proj",    # value projection
        "o_proj",    # output projection
        "gate_proj", # MLP gate
        "up_proj",   # MLP up
        "down_proj", # MLP down
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters : {trainable:,}  ({100 * trainable / total:.2f}% of total)")
print(f"Total parameters     : {total:,}")
print(f"\n✅ LoRA adapters injected into attention + MLP layers")


## 8 · Train

Expected time on free Colab T4:
- 500 samples  → ~15 minutes  (quick test)
- 3000 samples → ~60 minutes  (full run, recommended)

.


In [ ]:
OUTPUT_DIR = f"./abhilasha-qlora-{datetime.now().strftime('%Y%m%d-%H%M')}"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,     # effective batch = 2 × 4 = 8
    gradient_checkpointing=True,       # trade compute for memory
    optim="paged_adamw_32bit",         # paged optimizer — reduces VRAM spikes
    save_steps=100,
    logging_steps=20,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",                  # set to "wandb" if you want live loss charts
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

print("🚀 Starting training...")
print(f"   Samples   : {len(train_ds)}")
print(f"   Epochs    : 2")
print(f"   Output dir: {OUTPUT_DIR}")
print()

train_result = trainer.train()

print("\n✅ Training complete!")
print(f"   Final loss : {train_result.training_loss:.4f}")
print(f"   Runtime    : {train_result.metrics['train_runtime'] / 60:.1f} minutes")


## 9 · Evaluate Fine-Tuned Model

In [ ]:
model.config.use_cache = True  # re-enable for inference

print("Computing fine-tuned model perplexity on same 50 test samples...")
ft_ppl, ft_loss = compute_perplexity(model, tokenizer, test_ds, n_samples=50)

print("\n" + "=" * 50)
print("📊 RESULTS SUMMARY")
print("=" * 50)
print(f"Baseline perplexity   : {baseline_ppl:.2f}")
print(f"Fine-tuned perplexity : {ft_ppl:.2f}")
pct = ((baseline_ppl - ft_ppl) / baseline_ppl) * 100
direction = "improvement ✅" if pct > 0 else "regression ⚠️"
print(f"Change                : {pct:+.1f}% {direction}")
print("=" * 50)


## 10 · Before vs After — Qualitative Comparison

In [ ]:
print("=" * 70)
print("QUESTION:", TEST_QUESTION)
print("=" * 70)

print("\n🔴 BASE MODEL (before fine-tuning):")
print("-" * 40)
print(base_response)

print("\n🟢 ABHILASHA FINE-TUNED MODEL (after QLoRA):")
print("-" * 40)
ft_response = ask_model(model, tokenizer, TEST_QUESTION)
print(ft_response)
print("=" * 70)

# Additional test questions
extra_questions = [
    "What is the first-line treatment for Type 2 diabetes in a newly diagnosed patient?",
    "A 28-year-old woman presents with a butterfly-shaped rash on her face, joint pain, and fatigue. What is the most likely diagnosis?",
]

for q in extra_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask_model(model, tokenizer, q, max_new_tokens=200)}")
    print("-" * 60)


## 11 · Save Adapter Weights & Push to Hugging Face Hub

This pushes **only the LoRA adapter weights** (~50MB), not the full 7B model.  
Anyone who wants to use your model loads the base Mistral + your adapters.


In [ ]:
# Save adapter weights locally first
ADAPTER_DIR = f"./{NEW_MODEL}-adapters"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ Adapter weights saved to: {ADAPTER_DIR}")

# Push to Hub
REPO_ID = f"{HF_USERNAME}/{NEW_MODEL}"
trainer.model.push_to_hub(REPO_ID, use_auth_token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, use_auth_token=HF_TOKEN)
print(f"\n✅ Model pushed to: https://huggingface.co/{REPO_ID}")


## 12 · Generate Model Card (README for Hugging Face)

In [ ]:
model_card = f"""
language:
- en
license: apache-2.0
tags:
- medical
- qlora
- lora
- mistral
- fine-tuned
- peft
- healthcare
- nlp
base_model: mistralai/Mistral-7B-Instruct-v0.2
datasets:
- medalpaca/medical_meadow_medqa

# Abhilasha — Medical QA Fine-Tune (Mistral-7B + QLoRA)



## Model Summary

Fine-tuned `Mistral-7B-Instruct-v0.2` on the MedAlpaca Medical Meadow MedQA dataset using
**QLoRA (4-bit NF4 quantisation + LoRA adapters)** — entirely on a free Colab T4 GPU.

This adapter is designed to improve medical question-answering accuracy for clinical reasoning
tasks, and serves as the domain-specialised backbone for the Abhilasha multi-agent health assistant.

## Architecture Choices & Why

Choice | Reasoning
QLoRA (4-bit NF4) | Fits 7B model in ~14GB VRAM — enables free-GPU training
LoRA rank r=16 | Sweet spot: enough capacity for domain shift without overfitting on T4
Target: q/k/v/o + MLP projections | Attention + MLP together capture factual + reasoning shifts
Right-padding | Required for Mistral's positional encoding to work correctly
Cosine LR schedule | Smoother convergence than linear for instruction-tuned base

## Training Details

Parameter | Value |
Base model | mistralai/Mistral-7B-Instruct-v0.2
Dataset | medalpaca/medical_meadow_medqa
Training samples | {TRAIN_SAMPLES}
LoRA rank (r) | 16
LoRA alpha | 32
Epochs | 2
Batch size (effective) | 8 (2 × 4 grad accumulation)
Learning rate | 2e-4
Optimizer | paged_adamw_32bit
Hardware | NVIDIA T4 (free Colab)
Training time | ~60 minutes

## Results

| Metric | Base Model | Fine-Tuned | Delta |
Perplexity (medical QA) | {baseline_ppl:.2f} | {ft_ppl:.2f} | {((baseline_ppl - ft_ppl) / baseline_ppl * 100):+.1f}% |
NLL Loss | {baseline_loss:.4f} | {ft_loss:.4f} | {((baseline_loss - ft_loss) / baseline_loss * 100):+.1f}% |

# About Abhilasha

Abhilasha is a multi-agent AI healthcare assistant developed during the Ulster University
Agentic AI Hackathon (UK), selected for Round 2. This fine-tuned model serves as the
domain-specialised LLM backbone for its medical reasoning agent.



In [ ]:
from huggingface_hub import HfApi
api = HfApi()

api.upload_file(
    path_or_fileobj=f"{ADAPTER_DIR}/README.md",
    path_in_repo="README.md",
    repo_id=REPO_ID,
    token=HF_TOKEN,
)
print(f"✅ Model card pushed to https://huggingface.co/{REPO_ID}")
print(f" Done! Your model is live at:")
print(f"   https://huggingface.co/{REPO_ID}")


In [ ]:
pct_improvement = ((baseline_ppl - ft_ppl) / baseline_ppl) * 100
